# Module 3.1 — Containerization: guided HTTP test

This notebook checks the authentication and URL-shortener services through their HTTP APIs. It does not run Docker commands or access a Docker socket. Run cells in order, one at a time; do not use Run All.

## Learning goals
- Explain how a client registers and authenticates with the auth service.
- Use a bearer token to call protected shortener endpoints.
- Observe create, list, redirect, and update behavior.
- Verify that a mapping survives organizer-led container recreation.

## Before you begin
Your organizer must have prepared the keys and started both Compose services on your group VM. Obtain the VM address and the configured host ports. From this notebook, use that reachable address—not `localhost`, unless Jupyter is running on the same VM. Use disposable lab credentials and a harmless destination URL. Do not print or share your password or token.

The lab services use HTTP without TLS. Do not use real passwords or sensitive data. Keep this kernel running during the persistence checkpoint so the username and password variables remain available.

## 1. Prepare an HTTP client

The helper below uses only Python's standard library. It deliberately does not follow redirects, so a shortener response remains visible as HTTP 301 and the notebook does not fetch the destination website.

In [ ]:
import getpass
import json
import secrets
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from urllib.request import Request, HTTPRedirectHandler, build_opener

class DoNotFollowRedirects(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

http = build_opener(DoNotFollowRedirects())

def api_request(base_url, method, path='', payload=None, token=None):
    url = base_url.rstrip('/') + path
    headers = {'Accept': 'application/json'}
    body = None
    if payload is not None:
        headers['Content-Type'] = 'application/json'
        body = json.dumps(payload).encode('utf-8')
    if token is not None:
        headers['Authorization'] = 'Bearer ' + token
    request = Request(url, data=body, headers=headers, method=method)
    try:
        response = http.open(request, timeout=10)
    except HTTPError as error:
        response = error
    except URLError as error:
        raise RuntimeError(
            'Could not reach ' + url + '. Check the VM address, port, and service readiness.'
        ) from error
    try:
        status = response.code
        response_headers = response.headers
        raw_body = response.read()
    finally:
        response.close()
    try:
        data = json.loads(raw_body.decode('utf-8')) if raw_body else None
    except (UnicodeDecodeError, json.JSONDecodeError):
        data = None
    return status, response_headers, data

print('HTTP helper ready.')

### Predict
Before running the next cell: what status should the protected shortener root endpoint return when no bearer token is supplied? Why?

In [ ]:
AUTH_BASE = input('Auth service base URL, e.g. http://VM_ADDRESS:5001: ').strip().rstrip('/')
SHORTENER_BASE = input('Shortener service base URL, e.g. http://VM_ADDRESS:5000: ').strip().rstrip('/')

if not AUTH_BASE.startswith('http://') or not SHORTENER_BASE.startswith('http://'):
    raise ValueError('This practical is configured for HTTP URLs beginning with http://')

print('URLs recorded. No credentials or tokens have been sent yet.')

In [ ]:
status, headers, data = api_request(SHORTENER_BASE, 'GET', '/')
print('Unauthenticated GET / status:', status)
assert status == 403, 'Expected 403 for a protected endpoint without a token.'
print('Observed 403: the service is reachable and requires authentication.')

## 2. Register a disposable lab account

The username is generated for this notebook session. The password is requested without echoing it. If you rerun this cell in the same kernel, a 409 means that username already exists; continue to the login cell using the same password.

In [ ]:
if 'username' not in globals():
    username = 'lab_' + secrets.token_hex(4)
password = getpass.getpass('Enter a disposable lab password: ')
status, headers, data = api_request(
    AUTH_BASE, 'POST', '/users',
    {'username': username, 'password': password}
)
if status == 201:
    print('Account created. Username:', username)
elif status == 409:
    print('This session username already exists; try logging in with the password you just entered.')
else:
    raise RuntimeError('Registration returned HTTP ' + str(status) + '.')

### Observe and explain
What does HTTP 201 indicate? If you received 409, what does that tell you about a repeated registration request?

In [ ]:
status, headers, data = api_request(
    AUTH_BASE, 'POST', '/users/login',
    {'username': username, 'password': password}
)
assert status == 200, 'Login failed; check the username/password and registration result.'
token = data.get('token') if isinstance(data, dict) else None
assert token, 'The successful login response did not contain a token.'
print('Login succeeded. Token stored in memory; it is not displayed.')

## 3. Call the protected shortener API

Predict: how should the response differ when the same endpoint is called with a valid bearer token?

In [ ]:
status, headers, data = api_request(SHORTENER_BASE, 'GET', '/', token=token)
assert status == 200, 'Expected HTTP 200 from the authenticated shortener list endpoint.'
assert isinstance(data, dict) and isinstance(data.get('keys'), list), 'Expected a JSON object with a keys list.'
print('Authenticated request succeeded. Existing mapping count:', len(data['keys']))

## 4. Create and inspect a short URL

The destination below is a harmless example. The notebook will inspect the redirect response without following it.

In [ ]:
destination_url = 'https://example.org/'
status, headers, data = api_request(
    SHORTENER_BASE, 'POST', '/',
    {'url': destination_url}, token=token
)
assert status == 201, 'Expected HTTP 201 when creating a mapping.'
mapping_id = str(data['id'])
print('Mapping created. ID:', mapping_id)

In [ ]:
status, headers, data = api_request(SHORTENER_BASE, 'GET', '/', token=token)
assert status == 200 and mapping_id in data['keys'], 'The new mapping was not listed for this user.'
print('The protected list includes this mapping.')

status, headers, data = api_request(
    SHORTENER_BASE, 'GET', '/' + quote(mapping_id, safe='')
)
assert status == 301, 'Expected HTTP 301 from the public short URL.'
assert headers.get('Location') == destination_url, 'Redirect Location did not match the saved URL.'
print('Observed HTTP 301. Location:', headers.get('Location'))

## 5. Update the mapping

Predict what should change in the redirect after an owner updates the destination.

In [ ]:
updated_url = 'https://www.iana.org/domains/example'
mapping_path = '/' + quote(mapping_id, safe='')
status, headers, data = api_request(
    SHORTENER_BASE, 'PUT', mapping_path,
    {'url': updated_url}, token=token
)
assert status == 200, 'Expected HTTP 200 when the owner updates the mapping.'
status, headers, data = api_request(SHORTENER_BASE, 'GET', mapping_path)
assert status == 301 and headers.get('Location') == updated_url
print('Update verified: the redirect now points to the new destination.')

## 6. Persistence checkpoint — organizer action required

You have created a user and URL mapping. Pause here and ask the organizer to recreate the containers on this group VM. This interrupts the services, so do not do it while other participants are using them without coordination.

On the VM, from the directory containing `docker-compose.yml`, the organizer can run `docker compose down` and then `docker compose up -d`. **Do not add `-v`**: removing named volumes would delete the data this exercise is testing. Wait for the organizer to confirm both services are ready. Do not restart this notebook kernel.

### Predict
After the containers are recreated but the named volumes are retained, should the user and mapping still exist? What evidence will distinguish persisted data from data held only in a container's writable layer?

In [ ]:
password = getpass.getpass('Re-enter the same disposable lab password: ')
status, headers, data = api_request(
    AUTH_BASE, 'POST', '/users/login',
    {'username': username, 'password': password}
)
assert status == 200, 'Login after container recreation failed.'
token = data.get('token')
assert token, 'The login response did not contain a token.'
print('Login after recreation succeeded; the token remains hidden.')

In [ ]:
status, headers, data = api_request(SHORTENER_BASE, 'GET', '/', token=token)
assert status == 200, 'Could not list mappings after container recreation.'
assert mapping_id in data.get('keys', []), 'The mapping did not survive container recreation.'
status, headers, data = api_request(SHORTENER_BASE, 'GET', mapping_path)
assert status == 301 and headers.get('Location') == updated_url
print('Persistence verified: the mapping and updated destination survived container recreation.')

## 7. Clean up your test mapping

This removes only the mapping created in this notebook. Do not use the service's collection-wide delete endpoint. The disposable user remains in the auth volume; organizer-managed cleanup can remove workshop data after the session.

In [ ]:
status, headers, data = api_request(
    SHORTENER_BASE, 'DELETE', mapping_path, token=token
)
assert status == 204, 'Expected HTTP 204 when deleting the test mapping.'
print('The test mapping was deleted.')

## Explain what you observed
1. Why did the shortener reject the unauthenticated list request but accept it after login?
2. What did the HTTP 301 response and `Location` header show?
3. Which data survived container recreation, and why was retaining the named volumes important?
4. Why should the organizer avoid `docker compose down -v` during this test?

Record status codes and observations, but never include passwords or bearer tokens in screenshots or reports.